# Simple Baseline + Matchup Tests

This is a simple rule-based Mega Lucario ex baseline for the PTCG AI Battle Simulation competition.

The notebook writes `deck.csv`, writes a readable `main.py`, packages `submission.tar.gz`, and includes local matchup-test tables from my offline validation.

What changed in this version:

- baseline agent updated to `exp00014_20260617_202758`
- same compact Mega Lucario ex shell
- Crustle wall guard: avoid wasting Mega Lucario ex attacks into Crustle
- keep the non-ex Hariyama route available against Crustle
- no model weights, no internet, no external runtime dependency beyond the official competition data


## 1. Deck

The deck is written as a plain 60-line `deck.csv`, which is the format expected by this simulation competition.


In [1]:
from pathlib import Path

DECK = [
    673, 673, 674, 674, 675, 675, 676, 676,
    676, 677, 677, 677, 678, 678, 678, 678,
    1102, 1102, 1102, 1102, 1123, 1123, 1141, 1141,
    1141, 1141, 1142, 1142, 1142, 1142, 1152, 1152,
    6, 1159, 1182, 1182, 1192, 1192, 1192, 1192,
    1227, 1227, 1227, 1227, 6, 6, 6, 6,
    6, 6, 6, 6, 6, 6, 6, 6,
    6, 1182, 677, 1252,
]

Path("deck.csv").write_text("\n".join(map(str, DECK)) + "\n")
print("deck size:", len(DECK))
print("unique cards:", len(set(DECK)))


deck size: 60
unique cards: 17


## 2. Agent

The agent only receives legal options from the environment and returns option indices.
Most of the work is scoring legal options in a stable order. The Crustle-specific guard is intentionally small and rule-based.


In [2]:
%%writefile main.py
from __future__ import annotations

import os
from collections import defaultdict

from cg.api import (
    AreaType,
    Card,
    CardType,
    EnergyType,
    Observation,
    OptionType,
    Pokemon,
    SelectContext,
    all_card_data,
    to_observation_class,
)


class C:
    KYOGRE = 721
    SNOVER = 722
    MEGA_ABOMASNOW_EX = 723

    MAKUHITA = 673
    HARIYAMA = 674
    LUNATONE = 675
    SOLROCK = 676
    RIOLU = 677
    MEGA_LUCARIO_EX = 678

    BASIC_FIGHTING_ENERGY = 6
    DUSK_BALL = 1102
    SWITCH = 1123
    PREMIUM_POWER_PRO = 1141
    FIGHTING_GONG = 1142
    POKE_PAD = 1152
    HERO_CAPE = 1159
    BOSS_ORDERS = 1182
    CARMINE = 1192
    LILLIE_DETERMINATION = 1227
    GRAVITY_MOUNTAIN = 1252

    LUMIOSE_CITY = 1267
    LILLIES_PEARL = 1172
    LEGACY_ENERGY = 12


MEGA_BRAVE = 983
LOW_DECK_COUNT = 10


DECK_PATH = "deck.csv"
if not os.path.exists(DECK_PATH):
    DECK_PATH = "/kaggle_simulations/agent/deck.csv"
with open(DECK_PATH, "r", encoding="utf-8") as f:
    my_deck = [int(line) for line in f.read().splitlines() if line.strip()]


all_card = all_card_data()
card_table = {card.cardId: card for card in all_card}


class AttackPlan:
    def __init__(
        self,
        attacker: int = -1,
        target: int = -1,
        attack_index: int = -1,
        remain_hp: int = -1,
        needs_energy: bool = False,
    ):
        self.attacker = attacker
        self.target = target
        self.attack_index = attack_index
        self.remain_hp = remain_hp
        self.needs_energy = needs_energy


plan = AttackPlan()
pre_turn = -1
ability_used = False


def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    player = obs.current.players[player_index]
    match area:
        case AreaType.DECK:
            return obs.select.deck[index]
        case AreaType.HAND:
            return player.hand[index]
        case AreaType.DISCARD:
            return player.discard[index]
        case AreaType.ACTIVE:
            return player.active[index]
        case AreaType.BENCH:
            return player.bench[index]
        case AreaType.PRIZE:
            return player.prize[index]
        case AreaType.STADIUM:
            return obs.current.stadium[index]
        case AreaType.LOOKING:
            return obs.current.looking[index]
        case _:
            return None


def prize_count(pokemon: Pokemon) -> int:
    data = card_table[pokemon.id]
    count = 3 if data.megaEx else 2 if data.ex else 1
    for card in pokemon.energyCards:
        if card.id == C.LEGACY_ENERGY:
            count -= 1
    for card in pokemon.tools:
        if card.id == C.LILLIES_PEARL and "Lillie" in data.name:
            count -= 1
    return max(0, count)


def target_score(pokemon: Pokemon) -> int:
    data = card_table[pokemon.id]
    score = prize_count(pokemon) * 1000
    score += len(pokemon.energies) * 150
    score += len(pokemon.tools) * 100
    if data.stage2:
        score += 250
    elif data.stage1:
        score += 130

    if pokemon.id in {144, 322, 323, 337}:  # low-value support Pokemon
        score -= 200
    if pokemon.id == C.SNOVER:
        score += 950
    elif pokemon.id == C.MEGA_ABOMASNOW_EX:
        score += 250
    if pokemon.id == C.RIOLU:
        score += 800
    elif pokemon.id == C.MEGA_LUCARIO_EX:
        score += 100
    if pokemon.id == 112 and len(pokemon.energies) >= 1:  # Munkidori
        score += 300
    score += pokemon.hp
    return score


class LucarioPolicy:
    def __init__(self, obs: Observation):
        self.obs = obs
        self.state = obs.current
        self.select = obs.select
        self.context = self.select.context
        self.my_index = self.state.yourIndex
        self.op_index = 1 - self.my_index
        self.me = self.state.players[self.my_index]
        self.opponent = self.state.players[self.op_index]
        self.my_prizes_left = len(self.me.prize)

        self.field_counts = defaultdict(int)
        self.hand_counts = defaultdict(int)
        self.discard_counts = defaultdict(int)
        self.has_ready_lucario_line = False
        self.has_ready_hariyama_line = False
        self.can_switch = False
        self.can_gust = False
        self.can_attack = False
        self.can_use_mega_brave = False
        self.stadium_id = self.state.stadium[0].id if self.state.stadium else 0

        self._count_cards()
        self._scan_main_options()

    def choose(self) -> list[int]:
        if not self.select.option or self.select.maxCount == 0:
            return []

        if self.context == SelectContext.MAIN:
            self._plan_attack()

        scores = [self._score_option(option) for option in self.select.option]
        ranked = [i for i, _ in sorted(enumerate(scores), key=lambda item: item[1], reverse=True)]
        self._remember_lunatone_ability(ranked)
        return ranked[: self.select.maxCount]

    def _count_cards(self) -> None:
        for pokemon in self.me.active + self.me.bench:
            if pokemon is None:
                continue
            self.field_counts[pokemon.id] += 1
            if pokemon.id in {C.MAKUHITA, C.HARIYAMA} and len(pokemon.energies) >= 3:
                self.has_ready_hariyama_line = True
            if pokemon.id in {C.RIOLU, C.MEGA_LUCARIO_EX} and len(pokemon.energies) >= 2:
                self.has_ready_lucario_line = True

        for card in self.me.hand:
            self.hand_counts[card.id] += 1
        for card in self.me.discard:
            self.discard_counts[card.id] += 1

    def _scan_main_options(self) -> None:
        if self.context != SelectContext.MAIN:
            return
        for option in self.select.option:
            if option.type == OptionType.PLAY:
                card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
                if card.id == C.SWITCH:
                    self.can_switch = True
                elif card.id == C.BOSS_ORDERS:
                    self.can_gust = True
            elif option.type == OptionType.EVOLVE:
                card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
                if card.id == C.HARIYAMA:
                    self.can_gust = True
            elif option.type == OptionType.RETREAT:
                self.can_switch = True
            elif option.type == OptionType.ATTACK:
                self.can_attack = True
                if option.attackId == MEGA_BRAVE:
                    self.can_use_mega_brave = True

    def _my_board(self) -> list[Pokemon | None]:
        return self.me.active + self.me.bench

    def _opponent_board(self) -> list[Pokemon | None]:
        return self.opponent.active + self.opponent.bench

    def _opponent_has(self, ids: set[int]) -> bool:
        return any(pokemon is not None and pokemon.id in ids for pokemon in self._opponent_board())

    def _opponent_is_water_deck(self) -> bool:
        return self._opponent_has({C.KYOGRE, C.SNOVER, C.MEGA_ABOMASNOW_EX})

    def _opponent_is_crustle_wall(self) -> bool:
        return self._opponent_has({344, 345})

    def _can_evolve_board_index(self, board_index: int) -> bool:
        for option in self.select.option:
            if option.type != OptionType.EVOLVE:
                continue
            target_index = option.inPlayIndex
            if option.inPlayArea == AreaType.BENCH:
                target_index += 1
            if target_index == board_index:
                return True
        return False

    def _base_attack(self, pokemon: Pokemon, attack_index: int) -> tuple[int, int, int] | None:
        energy_required = 0
        base_damage = 0
        base_score = 0

        if pokemon.id == C.MEGA_LUCARIO_EX:
            if attack_index == 0:
                energy_required = 1
                base_damage = 130
                base_score += 60 * min(3, self.discard_counts[C.BASIC_FIGHTING_ENERGY])
            else:
                energy_required = 2
                base_damage = 270
            if self._opponent_is_water_deck() and len(self.opponent.prize) <= 3:
                base_score -= 500
        elif attack_index == 1:
            return None
        elif pokemon.id == C.HARIYAMA:
            energy_required = 3
            base_damage = 210
        elif pokemon.id == C.MAKUHITA:
            return None
        elif pokemon.id == C.SOLROCK and self.field_counts[C.LUNATONE] >= 1:
            energy_required = 1
            base_damage = 70

        if base_damage <= 0:
            return None
        return energy_required, base_damage, base_score

    def _base_attack_after_evolution(self, pokemon: Pokemon, board_index: int, attack_index: int):
        if pokemon.id == C.MAKUHITA and attack_index == 0 and self._can_evolve_board_index(board_index):
            return 3, 210, -100
        return self._base_attack(pokemon, attack_index)

    def _plan_attack(self) -> None:
        global plan
        best_score = -1
        plan = AttackPlan()

        if self.state.turn < 2:
            return

        for attacker_index, my_pokemon in enumerate(self._my_board()):
            if my_pokemon is None:
                continue
            if attacker_index != 0 and not self.can_switch:
                break

            for attack_index in range(2):
                attack = self._base_attack_after_evolution(my_pokemon, attacker_index, attack_index)
                if attack is None:
                    continue
                energy_required, base_damage, base_score = attack

                energy_count = len(my_pokemon.energies)
                if attack_index == 1 and attacker_index == 0 and energy_count >= 2 and not self.can_use_mega_brave:
                    break

                needs_energy = False
                if energy_count < energy_required:
                    if self.hand_counts[C.BASIC_FIGHTING_ENERGY] >= 1 and not self.state.energyAttached:
                        energy_count += 1
                        needs_energy = energy_count >= energy_required
                    if not needs_energy:
                        continue

                for target_index, op_pokemon in enumerate(self._opponent_board()):
                    if op_pokemon is None:
                        continue
                    if target_index != 0 and not self.can_gust:
                        break
                    if (
                        self._opponent_is_crustle_wall()
                        and my_pokemon.id == C.MEGA_LUCARIO_EX
                        and op_pokemon.id == 345
                    ):
                        continue

                    damage = base_damage
                    op_data = card_table[op_pokemon.id]
                    if op_data.weakness == EnergyType.FIGHTING:
                        damage *= 2
                    elif op_data.resistance == EnergyType.FIGHTING:
                        damage -= 30

                    score = target_score(op_pokemon)
                    prize = prize_count(op_pokemon) if op_pokemon.hp <= damage else 0
                    if prize == 0:
                        score *= damage / op_pokemon.hp
                    if len(self.opponent.prize) <= prize:
                        score = 50000

                    score += base_score
                    score += 220 if attacker_index == 0 else 0
                    score += 300 if target_index == 0 else 0
                    score += energy_count

                    if score > best_score:
                        best_score = score
                        plan = AttackPlan(
                            attacker=attacker_index,
                            target=target_index,
                            attack_index=attack_index,
                            remain_hp=op_pokemon.hp - damage,
                            needs_energy=needs_energy,
                        )

    def _energy_target_score(self, pokemon: Pokemon, active: bool) -> int:
        energy_count = len(pokemon.energies)
        score = 8000 + (10 if active else 0)

        if pokemon.id in {C.MAKUHITA, C.HARIYAMA}:
            score += 1 if pokemon.id == C.HARIYAMA else 0
            if self._opponent_is_crustle_wall():
                score += 260 if energy_count < 3 else 30
            else:
                score += 100 if energy_count < 3 else 0
                score -= 50 if self.has_ready_hariyama_line else 0
        elif pokemon.id == C.LUNATONE:
            score -= 100
        elif pokemon.id == C.SOLROCK:
            score += 20 if energy_count < 1 else -100
        elif pokemon.id in {C.RIOLU, C.MEGA_LUCARIO_EX}:
            score += 1 if pokemon.id == C.MEGA_LUCARIO_EX else 0
            score += 100 if energy_count < 2 else 0
            score -= 50 if self.has_ready_lucario_line else 0
        return score

    def _score_option(self, option) -> float:
        if option.type == OptionType.NUMBER:
            return option.number
        if option.type == OptionType.YES:
            return 100 if self.context == SelectContext.IS_FIRST else 1
        if option.type == OptionType.NO:
            return 0
        if option.type == OptionType.CARD:
            return self._score_card_choice(option)
        if option.type == OptionType.PLAY:
            return self._score_play(option)
        if option.type == OptionType.ATTACH:
            return self._score_attach(option)
        if option.type == OptionType.EVOLVE:
            return self._score_evolve(option)
        if option.type == OptionType.ABILITY:
            return self._score_ability(option)
        if option.type == OptionType.RETREAT:
            return 2000 if plan.attacker >= 1 else -1
        if option.type == OptionType.ATTACK:
            if (
                self._opponent_is_crustle_wall()
                and self.me.active
                and self.opponent.active
                and self.me.active[0].id == C.MEGA_LUCARIO_EX
                and self.opponent.active[0].id == 345
                and plan.target < 0
            ):
                return -1
            return 1100 if (option.attackId == MEGA_BRAVE) == (plan.attack_index == 1) else 1000
        return 0

    def _score_card_choice(self, option) -> float:
        card = get_card(self.obs, option.area, option.index, option.playerIndex)
        if card is None:
            return 0

        if self.context in {SelectContext.SWITCH, SelectContext.TO_ACTIVE}:
            return self._score_active_choice(option, card)
        if self.context == SelectContext.SETUP_ACTIVE_POKEMON:
            return self._score_setup_active(card)
        if self.context == SelectContext.TO_HAND:
            return self._score_to_hand(card)
        if self.context == SelectContext.ATTACH_FROM and isinstance(card, Pokemon):
            return self._energy_target_score(card, option.area == AreaType.ACTIVE)
        return 0

    def _score_active_choice(self, option, card: Pokemon | Card) -> float:
        if not isinstance(card, Pokemon):
            return 0

        if option.playerIndex != self.my_index:
            return 100 if option.index == plan.target - 1 else 0

        score = len(card.energies) * 2
        if option.index == plan.attacker - 1:
            score += 100
        if card.id == C.MEGA_LUCARIO_EX:
            score += 8 if self._opponent_is_water_deck() and len(self.opponent.prize) <= 3 else 20
        elif card.id == C.HARIYAMA and len(card.energies) >= 2:
            score += 45 if self._opponent_is_crustle_wall() else 15
        elif card.id == C.MAKUHITA and len(card.energies) >= 2:
            score += 35 if self._opponent_is_crustle_wall() else 10
        elif card.id == C.SOLROCK:
            score += 5
        elif card.id == C.RIOLU:
            score += 4
        return score

    def _score_setup_active(self, card: Pokemon | Card) -> int:
        if card.id == C.SOLROCK:
            return 2 if self.state.firstPlayer == self.my_index else 4
        if card.id == C.RIOLU:
            return 3
        if card.id == C.MAKUHITA:
            return 1
        return 0

    def _score_to_hand(self, card: Pokemon | Card) -> float:
        score = 200 - self.hand_counts[card.id] * 100
        if card.id == C.MAKUHITA:
            if self._opponent_is_crustle_wall():
                score += 80 if self.field_counts[card.id] < 2 else -20
            else:
                score += -10 if self.field_counts[card.id] >= 1 else 10
        elif card.id == C.HARIYAMA:
            if self._opponent_is_crustle_wall():
                score += 120 if self.field_counts[C.MAKUHITA] >= 1 else -5
            else:
                score += 20 if self.field_counts[C.MAKUHITA] >= 1 else -20
        elif card.id == C.LUNATONE:
            score += -250 if self.field_counts[card.id] >= 1 else 60
        elif card.id == C.SOLROCK:
            score += -250 if self.field_counts[card.id] >= 1 else 50
        elif card.id == C.RIOLU:
            lucario_line = self.field_counts[C.RIOLU] + self.field_counts[C.MEGA_LUCARIO_EX]
            score += -150 if lucario_line >= 2 else -3 if lucario_line >= 1 else 40
        elif card.id == C.MEGA_LUCARIO_EX:
            score += 40 if self.field_counts[C.RIOLU] >= 1 else -15
        elif card.id == C.BASIC_FIGHTING_ENERGY:
            score += 30 if not ability_used or not self.state.energyAttached else -1
        return score

    def _score_play(self, option) -> float:
        card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
        data = card_table[card.id]
        if data.cardType == CardType.POKEMON:
            return self._score_play_pokemon(card)
        return self._score_play_trainer(card)

    def _score_play_pokemon(self, card: Card) -> float:
        score = 20000
        if card.id in {C.LUNATONE, C.SOLROCK} and self.field_counts[card.id] >= 1:
            return -1
        if card.id == C.RIOLU and self.field_counts[C.RIOLU] + self.field_counts[C.MEGA_LUCARIO_EX] >= 2:
            return -1
        return score

    def _score_play_trainer(self, card: Card) -> float:
        if card.id == C.SWITCH:
            return 6000 if plan.attacker > 0 else -1
        if card.id == C.PREMIUM_POWER_PRO:
            if self.state.supporterPlayed and plan.remain_hp <= 0:
                return -1
            if not self.can_attack:
                can_bridge_draw = (
                    not self.state.supporterPlayed
                    and self.hand_counts[C.CARMINE] > 0
                    and self.hand_counts[C.LILLIE_DETERMINATION] == 0
                    and not self._low_deck()
                )
                return 3050 if can_bridge_draw else -1
            return 5000
        if card.id == C.BOSS_ORDERS:
            return 3200 if plan.target >= 1 else -1
        if card.id == C.CARMINE:
            return -1 if self._low_deck() else 3000
        if card.id == C.LILLIE_DETERMINATION:
            return -1 if self._low_deck() else 3100
        if card.id == C.GRAVITY_MOUNTAIN:
            return self._score_gravity_mountain()
        return 10000

    def _score_gravity_mountain(self) -> float:
        opponent_has_stage2 = any(
            pokemon is not None and card_table[pokemon.id].stage2 for pokemon in self._opponent_board()
        )
        if opponent_has_stage2:
            return 3500
        return 1200 if self.stadium_id else -1

    def _low_deck(self) -> bool:
        return self.me.deckCount <= LOW_DECK_COUNT

    def _score_attach(self, option) -> float:
        card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)
        pokemon = get_card(self.obs, option.inPlayArea, option.inPlayIndex, self.my_index)
        if not isinstance(pokemon, Pokemon):
            return 0

        if card.id == C.HERO_CAPE:
            score = 7000
            if self._opponent_is_water_deck():
                if pokemon.id == C.RIOLU:
                    return 12200
                if pokemon.id == C.MEGA_LUCARIO_EX:
                    return 12800
            if pokemon.id == C.RIOLU:
                score += 100
            elif pokemon.id == C.MEGA_LUCARIO_EX:
                score += 200
            return score

        score = self._energy_target_score(pokemon, option.inPlayArea == AreaType.ACTIVE)
        board_index = option.inPlayIndex if option.inPlayArea == AreaType.ACTIVE else option.inPlayIndex + 1
        if board_index == plan.attacker and plan.needs_energy:
            score += 200
        return score

    def _score_evolve(self, option) -> float:
        pokemon = get_card(self.obs, option.inPlayArea, option.inPlayIndex, self.my_index)
        if not isinstance(pokemon, Pokemon):
            return 0
        if pokemon.id == C.MAKUHITA and plan.target == 0 and not self._opponent_is_crustle_wall():
            return -1
        return 9000 + len(pokemon.energies)

    def _score_ability(self, option) -> float:
        card = get_card(self.obs, option.area, option.index, self.my_index)
        if card.id == C.LUMIOSE_CITY:
            return 1
        if card.id == C.LUNATONE and self._low_deck():
            return -1
        return 30000

    def _remember_lunatone_ability(self, ranked: list[int]) -> None:
        global ability_used
        if self.context != SelectContext.MAIN or not ranked:
            return
        option = self.select.option[ranked[0]]
        if option.type != OptionType.ABILITY:
            return
        card = get_card(self.obs, option.area, option.index, self.my_index)
        if card is not None and card.id == C.LUNATONE:
            ability_used = True


def agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    if obs.select is None:
        return my_deck

    global pre_turn
    global ability_used
    global plan

    if pre_turn != obs.current.turn:
        pre_turn = obs.current.turn
        ability_used = False
        plan = AttackPlan()

    return LucarioPolicy(obs).choose()


Writing main.py


## 3. Build Submission Bundle

The competition expects `main.py` and `deck.csv` at the top level of `submission.tar.gz`.
The official `cg` SDK is copied from the competition input before packaging.


In [3]:
import glob
import os
import shutil
import tarfile
from pathlib import Path


def find_cg_dir():
    candidates = [
        "../agent_submission/cg",
        "../../input/raw/pokemon-tcg-ai-battle/sample_submission/cg",
        "/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/cg",
        "/kaggle/input/pokemon-tcg-ai-battle/sample_submission/cg",
        "/kaggle/input/**/sample_submission/cg",
        "/kaggle/input/**/cg-lib/cg",
        "/kaggle/input/**/cg",
    ]
    for pattern in candidates:
        for path in glob.glob(pattern, recursive=True):
            if os.path.isdir(path) and os.path.exists(os.path.join(path, "api.py")):
                return path
    raise FileNotFoundError("Could not find the official cg SDK directory.")


cg_path = find_cg_dir()
if os.path.exists("cg"):
    shutil.rmtree("cg")
shutil.copytree(cg_path, "cg")

def add_clean_tree(tar, source, arcname):
    source = Path(source)
    for item in sorted(source.rglob("*")):
        rel = item.relative_to(source)
        if not item.is_file():
            continue
        if "__pycache__" in rel.parts or item.suffix in {".pyc", ".pyo"}:
            continue
        tar.add(item, arcname=str(Path(arcname) / rel))


with tarfile.open("submission.tar.gz", "w:gz") as tar:
    tar.add("main.py", arcname="main.py")
    tar.add("deck.csv", arcname="deck.csv")
    add_clean_tree(tar, "cg", "cg")

print("cg:", cg_path)
print("created:", Path("submission.tar.gz").resolve())


cg: /kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/cg
created: /kaggle/working/submission.tar.gz


## 4. Archive Check


In [4]:
import tarfile

with tarfile.open("submission.tar.gz", "r:gz") as tar:
    names = sorted(tar.getnames())

required = {"main.py", "deck.csv", "cg/api.py", "cg/libcg.so"}
missing = sorted(required - set(names))
if missing:
    raise RuntimeError(f"missing from archive: {missing}")
if any("__pycache__" in name or name.endswith((".pyc", ".pyo")) for name in names):
    raise RuntimeError("archive contains Python cache files")

print("\n".join(names[:12]))
print("archive files:", len(names))


cg/__init__.py
cg/api.py
cg/cg.dll
cg/game.py
cg/libcg.so
cg/sim.py
cg/utils.py
deck.csv
main.py
archive files: 9


## 5. Matchup Tests

These are local validation tables, not official leaderboard results.

Summary embedded from local runs:

- Crustle confirmation: `output/diagnostic_exp13_crustle_skip_ex_wall_public13_confirm_100_20260617_000001/summary/matchup_summary.csv`
- Public21 selected gate: `output/diagnostic_exp15_public21_selected_30_20260619_000001/summary/matchup_summary.csv`
- exp14 validation metric: `Crustle matchup win rate with public13 guarded regression review`
- Crustle matchup improvement in 100-game confirmation: `0.47` -> `0.77`
- Public21 selected mean win rate for exp14 rows: `0.7092`


In [5]:
CRUSTLE_CONFIRM_ROWS = [
  {
    "matchup": "exp13_base_vs_harukiharada_crustle",
    "games": "100",
    "completed_games": "100",
    "a_wins": "47",
    "b_wins": "53",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.47",
    "avg_turn": "21.13",
    "first_player_win_rate": "0.51",
    "median_final_prizes": "0.5"
  },
  {
    "matchup": "crustle_skip_ex_wall_vs_harukiharada_crustle",
    "games": "100",
    "completed_games": "100",
    "a_wins": "77",
    "b_wins": "23",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.77",
    "avg_turn": "16.8",
    "first_player_win_rate": "0.49",
    "median_final_prizes": "0.0"
  }
]
PUBLIC21_EXP14_ROWS = [
  {
    "matchup": "exp14_base_vs_lucario_sample_v1",
    "games": "30",
    "completed_games": "30",
    "a_wins": "18",
    "b_wins": "12",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.6",
    "avg_turn": "11.43",
    "first_player_win_rate": "0.5",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_validated_v2_public",
    "games": "30",
    "completed_games": "30",
    "a_wins": "14",
    "b_wins": "16",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.4667",
    "avg_turn": "11.77",
    "first_player_win_rate": "0.6333",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_official_random",
    "games": "30",
    "completed_games": "30",
    "a_wins": "29",
    "b_wins": "1",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.9667",
    "avg_turn": "5.6",
    "first_player_win_rate": "0.6667",
    "median_final_prizes": "4.5"
  },
  {
    "matchup": "exp14_base_vs_kiyota_dragapult",
    "games": "30",
    "completed_games": "30",
    "a_wins": "16",
    "b_wins": "14",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.5333",
    "avg_turn": "11.67",
    "first_player_win_rate": "0.5333",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_kiyota_iono",
    "games": "30",
    "completed_games": "30",
    "a_wins": "28",
    "b_wins": "2",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.9333",
    "avg_turn": "9.8",
    "first_player_win_rate": "0.5",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_kiyota_abomasnow",
    "games": "30",
    "completed_games": "30",
    "a_wins": "15",
    "b_wins": "15",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.5",
    "avg_turn": "9.8",
    "first_player_win_rate": "0.5333",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_zoli_dragapult",
    "games": "30",
    "completed_games": "30",
    "a_wins": "27",
    "b_wins": "3",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.9",
    "avg_turn": "11",
    "first_player_win_rate": "0.6",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_roman_strong_start_safe_lb860",
    "games": "30",
    "completed_games": "30",
    "a_wins": "13",
    "b_wins": "17",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.4333",
    "avg_turn": "12.2",
    "first_player_win_rate": "0.6",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_evgendvorkin_abomasnow",
    "games": "30",
    "completed_games": "30",
    "a_wins": "29",
    "b_wins": "0",
    "draws": "1",
    "errors": "0",
    "win_rate": "1.0",
    "avg_turn": "8.03",
    "first_player_win_rate": "0.4828",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_nursrijan_lucario",
    "games": "30",
    "completed_games": "30",
    "a_wins": "30",
    "b_wins": "0",
    "draws": "0",
    "errors": "0",
    "win_rate": "1.0",
    "avg_turn": "11.57",
    "first_player_win_rate": "0.5",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_harukiharada_crustle",
    "games": "30",
    "completed_games": "30",
    "a_wins": "24",
    "b_wins": "6",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.8",
    "avg_turn": "16.67",
    "first_player_win_rate": "0.6333",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_yakitori55_raging_bolt",
    "games": "30",
    "completed_games": "30",
    "a_wins": "29",
    "b_wins": "1",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.9667",
    "avg_turn": "9.37",
    "first_player_win_rate": "0.4667",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_pilkwang_lucario_v2",
    "games": "30",
    "completed_games": "30",
    "a_wins": "21",
    "b_wins": "9",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.7",
    "avg_turn": "12.07",
    "first_player_win_rate": "0.6667",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_biohack44_crustle_day1",
    "games": "30",
    "completed_games": "30",
    "a_wins": "16",
    "b_wins": "14",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.5333",
    "avg_turn": "14.3",
    "first_player_win_rate": "0.5",
    "median_final_prizes": "2.0"
  },
  {
    "matchup": "exp14_base_vs_biohack44_day2_new",
    "games": "30",
    "completed_games": "30",
    "a_wins": "24",
    "b_wins": "5",
    "draws": "1",
    "errors": "0",
    "win_rate": "0.8276",
    "avg_turn": "12.77",
    "first_player_win_rate": "0.8276",
    "median_final_prizes": "3.0"
  },
  {
    "matchup": "exp14_base_vs_pixiux_crustle_v1",
    "games": "30",
    "completed_games": "30",
    "a_wins": "16",
    "b_wins": "14",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.5333",
    "avg_turn": "14.73",
    "first_player_win_rate": "0.5333",
    "median_final_prizes": "2.0"
  },
  {
    "matchup": "exp14_base_vs_roman_crustle_lucario_v7_lb960",
    "games": "30",
    "completed_games": "30",
    "a_wins": "19",
    "b_wins": "11",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.6333",
    "avg_turn": "12.8",
    "first_player_win_rate": "0.4",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_kokinn_lucario_search_915",
    "games": "30",
    "completed_games": "30",
    "a_wins": "20",
    "b_wins": "10",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.6667",
    "avg_turn": "11.6",
    "first_player_win_rate": "0.3667",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_kacchan_lucario_anti_wall",
    "games": "30",
    "completed_games": "30",
    "a_wins": "22",
    "b_wins": "8",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.7333",
    "avg_turn": "11.7",
    "first_player_win_rate": "0.5667",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_penguin_public_915",
    "games": "30",
    "completed_games": "30",
    "a_wins": "14",
    "b_wins": "16",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.4667",
    "avg_turn": "13.1",
    "first_player_win_rate": "0.5",
    "median_final_prizes": "0.0"
  },
  {
    "matchup": "exp14_base_vs_yaroslav_lucario_crustle_aware",
    "games": "30",
    "completed_games": "30",
    "a_wins": "21",
    "b_wins": "9",
    "draws": "0",
    "errors": "0",
    "win_rate": "0.7",
    "avg_turn": "12.27",
    "first_player_win_rate": "0.6",
    "median_final_prizes": "0.0"
  }
]
EXP14_METRICS = {
  "cv_score": 0.77,
  "metric": "Crustle matchup win rate with public13 guarded regression review",
  "fold_strategy": "local public13 notebook matchup suite plus targeted Crustle confirmation",
  "screen": {
    "run_dir": "output/diagnostic_exp13_targeted_screen_30_20260617_000001",
    "baseline_agent": "exp13_base",
    "candidate_agent": "crustle_skip_ex_wall",
    "games_per_matchup": 30,
    "matchups": 13,
    "candidate_mean_win_rate": 0.7943,
    "mean_delta_vs_baseline": 0.0455,
    "worst_delta_vs_baseline": -0.0667,
    "crustle_delta": 0.4333,
    "errors": 0
  },
  "confirmation": {
    "run_dir": "output/diagnostic_exp13_crustle_skip_ex_wall_public13_confirm_100_20260617_000001",
    "comparison_dir": "output/diagnostic_exp13_crustle_skip_ex_wall_public13_confirm_100_20260617_000001/comparison",
    "loss_analysis_dir": "output/diagnostic_exp13_crustle_skip_ex_wall_public13_confirm_100_20260617_000001/loss_crustle_skip_ex_wall",
    "baseline_agent": "exp13_base",
    "candidate_agent": "crustle_skip_ex_wall",
    "games_per_matchup": 100,
    "matchups": 13,
    "baseline_mean_win_rate": 0.7597,
    "candidate_mean_win_rate": 0.7586,
    "mean_delta_vs_baseline": -0.0011,
    "worst_observed_delta_vs_baseline": -0.098,
    "baseline_crustle_win_rate": 0.47,
    "candidate_crustle_win_rate": 0.77,
    "crustle_delta": 0.3,
    "candidate_errors": 0
  },
  "interpretation": "The code change is guarded by visible Dwebble/Crustle, so non-Crustle same-run deltas are treated as simulator sampling noise. The targeted Crustle improvement is large and causally explained by Crustle's Pokemon-ex damage prevention.",
  "notes": "Deck unchanged from exp00013. This candidate is a metagame patch, not a broad Public13 mean upgrade."
}

try:
    import pandas as pd
    print("Crustle-focused 100-game confirmation")
    display(pd.DataFrame(CRUSTLE_CONFIRM_ROWS))
    print("Public21 selected gate: exp14 rows")
    display(pd.DataFrame(PUBLIC21_EXP14_ROWS))
except Exception:
    print("Crustle-focused 100-game confirmation")
    for row in CRUSTLE_CONFIRM_ROWS:
        print(row)
    print("Public21 selected gate: exp14 rows")
    for row in PUBLIC21_EXP14_ROWS:
        print(row)

print("exp14 metrics:")
print(EXP14_METRICS)


Crustle-focused 100-game confirmation


,matchup,games,completed_games,a_wins,b_wins,draws,errors,win_rate,avg_turn,first_player_win_rate,median_final_prizes
0,exp13_base_vs_harukiharada_crustle,100,100,47,53,0,0,0.47,21.13,0.51,0.5
1,crustle_skip_ex_wall_vs_harukiharada_crustle,100,100,77,23,0,0,0.77,16.8,0.49,0.0


Public21 selected gate: exp14 rows


,matchup,games,completed_games,a_wins,b_wins,draws,errors,win_rate,avg_turn,first_player_win_rate,median_final_prizes
0,exp14_base_vs_lucario_sample_v1,30,30,18,12,0,0,0.6,11.43,0.5,0.0
1,exp14_base_vs_validated_v2_public,30,30,14,16,0,0,0.4667,11.77,0.6333,0.0
2,exp14_base_vs_official_random,30,30,29,1,0,0,0.9667,5.6,0.6667,4.5
3,exp14_base_vs_kiyota_dragapult,30,30,16,14,0,0,0.5333,11.67,0.5333,0.0
4,exp14_base_vs_kiyota_iono,30,30,28,2,0,0,0.9333,9.8,0.5,0.0
5,exp14_base_vs_kiyota_abomasnow,30,30,15,15,0,0,0.5,9.8,0.5333,0.0
6,exp14_base_vs_zoli_dragapult,30,30,27,3,0,0,0.9,11,0.6,0.0
7,exp14_base_vs_roman_strong_start_safe_lb860,30,30,13,17,0,0,0.4333,12.2,0.6,0.0
8,exp14_base_vs_evgendvorkin_abomasnow,30,30,29,0,1,0,1.0,8.03,0.4828,0.0
9,exp14_base_vs_nursrijan_lucario,30,30,30,0,0,0,1.0,11.57,0.5,0.0


exp14 metrics:
{'cv_score': 0.77, 'metric': 'Crustle matchup win rate with public13 guarded regression review', 'fold_strategy': 'local public13 notebook matchup suite plus targeted Crustle confirmation', 'screen': {'run_dir': 'output/diagnostic_exp13_targeted_screen_30_20260617_000001', 'baseline_agent': 'exp13_base', 'candidate_agent': 'crustle_skip_ex_wall', 'games_per_matchup': 30, 'matchups': 13, 'candidate_mean_win_rate': 0.7943, 'mean_delta_vs_baseline': 0.0455, 'worst_delta_vs_baseline': -0.0667, 'crustle_delta': 0.4333, 'errors': 0}, 'confirmation': {'run_dir': 'output/diagnostic_exp13_crustle_skip_ex_wall_public13_confirm_100_20260617_000001', 'comparison_dir': 'output/diagnostic_exp13_crustle_skip_ex_wall_public13_confirm_100_20260617_000001/comparison', 'loss_analysis_dir': 'output/diagnostic_exp13_crustle_skip_ex_wall_public13_confirm_100_20260617_000001/loss_crustle_skip_ex_wall', 'baseline_agent': 'exp13_base', 'candidate_agent': 'crustle_skip_ex_wall', 'games_per_matchu

## 6. Small Smoke Test

This is only a quick local sanity check against a random agent. The real validation is the matchup table above and the Kaggle validation episode after submission.


In [6]:
import importlib.util
import random
import sys

sys.path.insert(0, ".")

spec = importlib.util.spec_from_file_location("simple_baseline", "main.py")
simple_baseline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simple_baseline)

from cg.api import to_observation_class
from cg.game import battle_finish, battle_select, battle_start


def random_agent(obs_dict):
    obs = to_observation_class(obs_dict)
    if obs.select is None:
        return DECK
    return random.sample(range(len(obs.select.option)), obs.select.maxCount)


def play_game(agent0, deck0, agent1, deck1, max_steps=1000):
    obs = None
    try:
        obs, start_data = battle_start(deck0, deck1)
        if obs is None:
            return None, 0, f"battle_start_failed:{start_data.errorPlayer}:{start_data.errorType}"
        for step in range(max_steps + 1):
            obc = to_observation_class(obs)
            if obc.current.result >= 0:
                return obc.current.result, step, ""
            active_agent = agent0 if obc.current.yourIndex == 0 else agent1
            obs = battle_select(active_agent(obs))
        return None, max_steps, "max_steps"
    except Exception as exc:
        return None, 0, f"{type(exc).__name__}:{exc}"
    finally:
        if obs is not None:
            battle_finish()


N_GAMES = 6
summary = {"baseline_wins": 0, "random_wins": 0, "draws": 0, "errors": 0}
for game in range(N_GAMES):
    if game % 2 == 0:
        result, steps, error = play_game(simple_baseline.agent, DECK, random_agent, DECK)
        baseline_index = 0
    else:
        result, steps, error = play_game(random_agent, DECK, simple_baseline.agent, DECK)
        baseline_index = 1
    if error:
        summary["errors"] += 1
    elif result == 2:
        summary["draws"] += 1
    elif result == baseline_index:
        summary["baseline_wins"] += 1
    else:
        summary["random_wins"] += 1

summary


{'baseline_wins': 6, 'random_wins': 0, 'draws': 0, 'errors': 0}